# Ticket Demand Forecasting Model

Trains a time-series regression model to forecast daily ticket demand using features from `ML_FEATURES.ML_TICKET_DEMAND_FEATURES`.

**Target:** `daily_reserved` (number of tickets reserved per day per ticket type)

**Approach:** XGBoost regressor with lag features and calendar attributes, registered to Snowflake Model Registry

In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd
import numpy as np

session = get_active_session()
print(f"Connected: {session.get_current_database()}.{session.get_current_schema()}")

In [ ]:
%%sql -r demand_data
SELECT *
FROM NS11MM_DW_DEV.ML_FEATURES.ML_TICKET_DEMAND_FEATURES
WHERE ENTRY_DATE IS NOT NULL
ORDER BY TICKET_TYPE, ENTRY_DATE

In [ ]:
df = demand_data
print(f"Dataset shape: {df.shape}")
print(f"Date range: {df['ENTRY_DATE'].min()} to {df['ENTRY_DATE'].max()}")
print(f"Ticket types: {df['TICKET_TYPE'].nunique()}")
print(df['TICKET_TYPE'].value_counts())

In [ ]:
feature_cols = [
    'DAY_OF_WEEK_NUM',
    'IS_WEEKEND',
    'MONTH_NUM',
    'RESERVED_LAG_1D',
    'RESERVED_LAG_7D',
    'RESERVED_7D_AVG',
    'RESERVED_30D_AVG',
    'RESERVED_30D_STDDEV',
    'DAILY_CAPACITY',
    'DAILY_UTILIZATION_PCT',
    'WINDOWS_SOLD_OUT',
    'WINDOWS_HIGH_DEMAND',
    'TOTAL_WINDOWS'
]

target_col = 'DAILY_RESERVED'

df_model = df[feature_cols + [target_col, 'ENTRY_DATE', 'TICKET_TYPE']].dropna(subset=feature_cols)

df_model['IS_WEEKEND'] = df_model['IS_WEEKEND'].astype(int)

cutoff_date = df_model['ENTRY_DATE'].max() - pd.Timedelta(days=30)
train = df_model[df_model['ENTRY_DATE'] <= cutoff_date]
test = df_model[df_model['ENTRY_DATE'] > cutoff_date]

X_train = train[feature_cols].fillna(0)
y_train = train[target_col]
X_test = test[feature_cols].fillna(0)
y_test = test[target_col]

print(f"Train: {X_train.shape[0]} rows ({train['ENTRY_DATE'].min()} to {train['ENTRY_DATE'].max()})")
print(f"Test:  {X_test.shape[0]} rows ({test['ENTRY_DATE'].min()} to {test['ENTRY_DATE'].max()})")

In [ ]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

model = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
mape = np.mean(np.abs((y_test - y_pred) / np.maximum(y_test, 1))) * 100

print(f"MAE:  {mae:.1f} tickets")
print(f"RMSE: {rmse:.1f} tickets")
print(f"R²:   {r2:.4f}")
print(f"MAPE: {mape:.1f}%")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(12, 8))

test_plot = test.copy()
test_plot['predicted'] = y_pred
sample_type = test_plot['TICKET_TYPE'].iloc[0]
type_data = test_plot[test_plot['TICKET_TYPE'] == sample_type]

axes[0].plot(type_data['ENTRY_DATE'], type_data[target_col], label='Actual', marker='o', markersize=3)
axes[0].plot(type_data['ENTRY_DATE'], type_data['predicted'], label='Predicted', marker='x', markersize=3)
axes[0].set_title(f'Demand Forecast vs Actual — {sample_type}')
axes[0].legend()
axes[0].set_ylabel('Tickets Reserved')

feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

axes[1].barh(feature_importance['feature'], feature_importance['importance'])
axes[1].set_xlabel('Feature Importance')
axes[1].set_title('Feature Importance for Demand Forecasting')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
from snowflake.ml.registry import Registry

reg = Registry(session=session, database_name='NS11MM_DW_DEV', schema_name='ML_REGISTRY')

metrics = {
    'mae': float(mae),
    'rmse': float(rmse),
    'r2': float(r2),
    'mape': float(mape),
    'train_rows': int(X_train.shape[0]),
    'test_rows': int(X_test.shape[0]),
    'n_features': len(feature_cols)
}

mv = reg.log_model(
    model=model,
    model_name='ticket_demand_xgboost',
    version_name='v1',
    sample_input_data=X_test.head(10),
    metrics=metrics,
    comment='XGBoost ticket demand forecaster trained on ML_TICKET_DEMAND_FEATURES'
)

print(f"Model registered: {mv.model_name} v{mv.version_name}")
print(f"Metrics: {metrics}")